<a href="https://colab.research.google.com/github/yassinemaataoui/Colab_project/blob/main/TD_04_Big_Data_Correction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!apt-get update -qq && apt-get install -y openjdk-11-jdk
version = "3.5.6"
hadoop_version = "hadoop3"
url = f"https://downloads.apache.org/spark/spark-{version}/spark-{version}-bin-{hadoop_version}.tgz"
!wget {url} -O spark-{version}-bin-{hadoop_version}.tgz
!ls -lh spark-{version}-bin-{hadoop_version}.tgz
!tar -xvzf spark-{version}-bin-{hadoop_version}.tgz
!pip install -q findspark
import os, findspark
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["SPARK_HOME"] = f"/content/spark-{version}-bin-{hadoop_version}"
findspark.init()
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Atelier_PySpark").getOrCreate()
spark

In [ ]:
from google.colab import files
print("⬆Veuillez uploader le fichier consommation_ecowatt.csv")
uploaded = files.upload()

⬆Veuillez uploader le fichier consommation_ecowatt.csv


Saving consommation_ecowatt.csv to consommation_ecowatt (1).csv


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date, date_format, year, month, hour, sum as _sum, avg, row_number, when
from pyspark.sql.window import Window
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


spark = SparkSession.builder.appName("EcoWattSpark").getOrCreate()

# -------------------------------
#  Exercice 1 – Charger et explorer le dataset
# -------------------------------
print("Exercice 1 – Exploration du dataset")
df = spark.read.csv("consommation_ecowatt.csv", header=True, inferSchema=True)
print("Schéma :")
df.printSchema()
print("Nombre total de mesures :", df.count())
df.show(5)

Exercice 1 – Exploration du dataset
Schéma :
root
 |-- id_mesure: integer (nullable = true)
 |-- date_heure: timestamp (nullable = true)
 |-- ville: string (nullable = true)
 |-- type_batiment: string (nullable = true)
 |-- conso_kwh: double (nullable = true)
 |-- temperature_ext: double (nullable = true)
 |-- cout_kwh: double (nullable = true)
 |-- nb_personnes: integer (nullable = true)

Nombre total de mesures : 300
+---------+-------------------+----------+-------------+---------+---------------+--------+------------+
|id_mesure|         date_heure|     ville|type_batiment|conso_kwh|temperature_ext|cout_kwh|nb_personnes|
+---------+-------------------+----------+-------------+---------+---------------+--------+------------+
|        1|2025-02-05 20:00:00| Marrakech|  Résidentiel|   7973.1|           35.3|    1.37|         115|
|        2|2025-02-23 22:00:00|    Agadir|        École|   2096.1|           19.6|     1.3|         199|
|        3|2025-02-17 02:00:00|    Tanger|        Éc

In [ ]:
# -------------------------------
# Exercice 2 – Nettoyage et enrichissement
# -------------------------------
print("\n Exercice 2 – Nettoyage et enrichissement")
df_clean = df.dropna()
df_clean = df_clean.filter(
    (col("conso_kwh") >= 0) & (col("conso_kwh") <= 20000) &
    (col("temperature_ext") >= -10) & (col("temperature_ext") <= 50)
)
df_clean = df_clean.withColumn("cout_total", col("conso_kwh") * col("cout_kwh")) \
                   .withColumn("conso_par_personne", col("conso_kwh") / col("nb_personnes"))
print("Nombre de lignes après nettoyage :", df_clean.count())
df_clean.show(5)


 Exercice 2 – Nettoyage et enrichissement
Nombre de lignes après nettoyage : 300
+---------+-------------------+----------+-------------+---------+---------------+--------+------------+----------+------------------+
|id_mesure|         date_heure|     ville|type_batiment|conso_kwh|temperature_ext|cout_kwh|nb_personnes|cout_total|conso_par_personne|
+---------+-------------------+----------+-------------+---------+---------------+--------+------------+----------+------------------+
|        1|2025-02-05 20:00:00| Marrakech|  Résidentiel|   7973.1|           35.3|    1.37|         115| 10923.147| 69.33130434782609|
|        2|2025-02-23 22:00:00|    Agadir|        École|   2096.1|           19.6|     1.3|         199|   2724.93|10.533165829145728|
|        3|2025-02-17 02:00:00|    Tanger|        École|   1720.2|           34.9|     1.2|         167|   2064.24| 10.30059880239521|
|        4|2025-02-15 15:00:00|Casablanca|      Hôpital|   1726.2|           11.1|    1.23|         130|  21

In [ ]:
# -------------------------------
# Exercice 3 – Analyses descriptives
# -------------------------------
print("\n Exercice 3 – Consommation par ville et type de bâtiment")

print("\n Consommation totale par ville")
df_ville = df_clean.groupBy("ville").agg(_sum("conso_kwh").alias("conso_totale")).orderBy(col("conso_totale").desc())
df_ville.show()

print("\n Consommation moyenne par type de bâtiment")
df_type = df_clean.groupBy("type_batiment").agg(avg("conso_kwh").alias("conso_moyenne")).orderBy(col("conso_moyenne").desc())
df_type.show()


 Exercice 3 – Consommation par ville et type de bâtiment

 Consommation totale par ville
+----------+------------------+
|     ville|      conso_totale|
+----------+------------------+
|    Tanger|          335346.9|
|    Agadir|257718.90000000005|
|     Rabat|244939.20000000004|
| Marrakech|232741.69999999998|
|Casablanca|217569.00000000003|
|       Fès|209143.39999999997|
+----------+------------------+


 Consommation moyenne par type de bâtiment
+-------------+-----------------+
|type_batiment|    conso_moyenne|
+-------------+-----------------+
|  Résidentiel|5268.075000000002|
|       Bureau|5156.969841269843|
|        École|5112.007547169812|
|      Hôpital|4711.804918032788|
|   Industriel|4695.876271186441|
+-------------+-----------------+



In [ ]:
# -------------------------------
#  Exercice 4 – Analyse temporelle
# -------------------------------
print("\n Exercice 4 – Analyse temporelle")
df_time = df_clean.withColumn("date", to_date("date_heure")) \
                  .withColumn("heure", hour("date_heure")) \
                  .withColumn("jour", date_format(to_date("date_heure"), "EEEE")) \
                  .withColumn("mois", month("date_heure"))

print("\n Consommation totale par mois")
df_time.groupBy("mois").agg(_sum("conso_kwh").alias("conso_totale")).orderBy("mois").show()

print("\n Consommation moyenne par jour")
df_time.groupBy("jour").agg(avg("conso_kwh").alias("conso_moyenne")).show()

print("\n Consommation horaire moyenne")
df_time.groupBy("heure").agg(avg("conso_kwh").alias("conso_moyenne")).orderBy("heure").show()


 Exercice 4 – Analyse temporelle

 Consommation totale par mois
+----+------------------+
|mois|      conso_totale|
+----+------------------+
|   1|          457750.5|
|   2|488114.30000000005|
|   3|          551594.3|
+----+------------------+


 Consommation moyenne par jour
+---------+-----------------+
|     jour|    conso_moyenne|
+---------+-----------------+
|Wednesday|5227.132692307694|
|  Tuesday|5118.782926829269|
|   Friday|4649.504761904762|
| Thursday|4903.738636363636|
| Saturday|4463.758064516128|
|   Monday|5002.612500000001|
|   Sunday|5476.811764705883|
+---------+-----------------+


 Consommation horaire moyenne
+-----+------------------+
|heure|     conso_moyenne|
+-----+------------------+
|    0|           6133.05|
|    1|            3642.4|
|    2| 4624.241666666667|
|    3| 3408.207692307692|
|    4|           4969.84|
|    5| 3913.891666666666|
|    6|  5513.11818181818|
|    7| 6501.415384615385|
|    8| 5972.026666666668|
|    9| 6239.542857142857|
|   10|

In [ ]:
# -------------------------------
#  Exercice 6 – Requêtes Spark SQL
# -------------------------------
print("\n Exercice 6 – Spark SQL")
df_clean.createOrReplaceTempView("conso")

print("\n Consommation totale par type de bâtiment")
spark.sql("""
    SELECT type_batiment, SUM(conso_kwh) AS conso_totale
    FROM conso
    GROUP BY type_batiment
    ORDER BY conso_totale DESC
""").show()

print("\n Consommation moyenne par personne")
spark.sql("""
    SELECT AVG(conso_par_personne) AS conso_moyenne_par_personne
    FROM conso
""").show()

print("\n Top 2 bâtiments énergivores par ville")
spark.sql("""
    SELECT ville, type_batiment, conso_kwh
    FROM (
        SELECT ville, type_batiment, conso_kwh,
               ROW_NUMBER() OVER(PARTITION BY ville ORDER BY conso_kwh DESC) AS rank
        FROM conso
    ) tmp
    WHERE rank <= 2
""").show()


 Exercice 6 – Spark SQL

 Consommation totale par type de bâtiment
+-------------+-----------------+
|type_batiment|     conso_totale|
+-------------+-----------------+
|  Résidentiel|337156.8000000001|
|       Bureau|324889.1000000001|
|      Hôpital|287420.1000000001|
|   Industriel|         277056.7|
|        École|         270936.4|
+-------------+-----------------+


 Consommation moyenne par personne
+--------------------------+
|conso_moyenne_par_personne|
+--------------------------+
|          79.4067845188413|
+--------------------------+


 Top 2 bâtiments énergivores par ville
+----------+-------------+---------+
|     ville|type_batiment|conso_kwh|
+----------+-------------+---------+
|    Agadir|      Hôpital|   9404.3|
|    Agadir|        École|   9350.2|
|Casablanca|  Résidentiel|   9466.5|
|Casablanca|        École|   9448.2|
|       Fès|  Résidentiel|   9507.3|
|       Fès|      Hôpital|   9478.1|
| Marrakech|        École|   9969.1|
| Marrakech|       Bureau|   9877

* **ROW_NUMBER() OVER(PARTITION BY ville ORDER BY conso_kwh DESC)** : crée un numéro d’ordre (rank) pour chaque bâtiment selon sa consommation.
* **PARTITION BY ville** : le classement est refait séparément pour chaque ville.
* **ORDER BY conso_kwh DESC** : les bâtiments sont triés du plus grand au plus petit consommateur d’énergie.

* **WHERE rank <= 2** : garde uniquement les deux premiers bâtiments (rang 1 et 2) dans chaque ville.

In [ ]:
# -------------------------------
#  Exercice 7 – Agrégations conditionnelles
# -------------------------------
print("\n Exercice 7 – Agrégations conditionnelles")
df_cond = df_clean.withColumn(
    "niveau_conso",
    when(col("conso_kwh") < 500, "Faible").when((col("conso_kwh") >= 500) & (col("conso_kwh") < 2000), "Moyenne").otherwise("Élevée")
)

df_cond.groupBy("niveau_conso", "type_batiment").agg( _sum("cout_total").alias("cout_total"),
    avg("conso_kwh").alias("conso_moyenne") ).orderBy("niveau_conso").show()


 Exercice 7 – Agrégations conditionnelles
+------------+-------------+------------------+------------------+
|niveau_conso|type_batiment|        cout_total|     conso_moyenne|
+------------+-------------+------------------+------------------+
|      Faible|       Bureau|           258.431|             228.7|
|      Faible|        École|          1484.511|           329.175|
|      Faible|   Industriel|1333.1490000000001|383.26666666666665|
|      Faible|  Résidentiel|1255.7250000000001|             356.5|
|      Faible|      Hôpital|2033.6349999999998|             307.7|
|     Moyenne|  Résidentiel|         17867.157| 1373.427272727273|
|     Moyenne|   Industriel|           6773.94|            1061.5|
|     Moyenne|      Hôpital|         18554.281|           1470.39|
|     Moyenne|        École|           13567.9|1581.7571428571428|
|     Moyenne|       Bureau|         24419.622|1448.7692307692307|
|      Élevée|      Hôpital|325671.29399999994| 5895.167391304348|
|      Élevée|     